# Reverse kinematics complète du robot

Ce notebook suit les mêmes étapes que `forward_kinematic_complete.ipynb`, dans le sens inverse :

**pose du triangle + orientations des biellettes → positions des effecteurs des 5-barres → angles des 6 moteurs.**

Il y a six angles moteurs, mais la plateforme triangulaire a seulement trois degrés de liberté. La pose du triangle seule ne suffit donc pas à définir une solution unique. On fixe aussi les trois orientations des biellettes de connexion (du sommet du triangle vers l'effecteur de la 5-barre), puis on choisit parmi les branches géométriques celle qui est la plus proche d'une posture de référence.

Les longueurs et les positions des points fixes ci-dessous reprennent les conventions du notebook de cinématique directe. Les angles sont en radians et les longueurs en mètres.

## 1 — Paramètres géométriques et pose désirée

La pose est définie par le sommet `A=(x_A,y_A)` et l'angle `alpha` du côté `AB`. Les angles `connector_angles` fixent l'orientation de chaque biellette entre un sommet et l'effecteur correspondant.

In [1]:
import math
import numpy as np
import matplotlib.pyplot as plt

# Longueurs des 5-barres (m), identiques au notebook de cinématique directe
AB = BC = CD = DE = EA = CF = FF = 0.2
l_triangle = 0.2  # biellette effecteur-sommet
d_triangle = 0.2  # côté du triangle équilatéral

# Pose désirée (m, radians) : exemple construit depuis theta_i = pi/2
x_A = 0.2985946040531197
y_A = 0.39687327124506444
alpha = 1.2150299244288536

# Direction sommet -> effecteur pour chacune des trois jambes
connector_angles = np.array([
    -3.022973726504686,
     2.4454863999368217,
     2.2003671467217503,
])

# Posture de référence de chaque paire de moteurs (radians)
reference_angles = np.tile([np.pi / 2, np.pi / 2], (3, 1))

if connector_angles.shape != (3,) or reference_angles.shape != (3, 2):
    raise ValueError("Il faut trois angles de connexion et trois paires d'angles de référence.")

## 2 — Sommets désirés et effecteurs des 5-barres

On reconstruit les sommets du triangle, puis chaque effecteur se trouve à une distance `l_triangle` de son sommet, dans la direction de connexion choisie.

In [2]:
A = np.array([x_A, y_A], dtype=float)
B = A + d_triangle * np.array([np.cos(alpha), np.sin(alpha)])
C = A + d_triangle * np.array([np.cos(alpha + np.pi / 3),
                                np.sin(alpha + np.pi / 3)])
platform_vertices = np.array([A, B, C])

connector_vectors = l_triangle * np.column_stack((
    np.cos(connector_angles), np.sin(connector_angles)
))
effectors_global = platform_vertices + connector_vectors

print("Sommets désirés A, B, C (m) :")
for name, point in zip(("A", "B", "C"), platform_vertices):
    print(f"{name} = ({point[0]:.6f}, {point[1]:.6f})")

print("\nEffecteurs désirés (m) :")
for i, point in enumerate(effectors_global, start=1):
    print(f"E{i} = ({point[0]:.6f}, {point[1]:.6f})")

Sommets désirés A, B, C (m) :
A = (0.298595, 0.396873)
B = (0.368256, 0.584349)
C = (0.171067, 0.550940)

Effecteurs désirés (m) :
E1 = (0.100000, 0.373205)
E2 = (0.214787, 0.712596)
E3 = (0.053307, 0.712596)


## 3 — Repères locaux des mécanismes

Chaque 5-barre est résolue dans son repère local : origine au moteur gauche et axe `x` dirigé vers le moteur droit. On transforme les effecteurs globaux dans ces trois repères avant de calculer les angles.

In [3]:
# Positions fixes des moteurs, reprises de compute_fixed_points() dans le notebook FK
A1 = np.array([0.0, 0.0])
A2 = np.array([
    (0.5 + np.cos(np.pi / 3) + 2 * np.cos(np.pi / 6)
     + np.cos(5 * np.pi / 6 - 2 * np.pi / 5)) * 0.2,
    (np.sin(2 * np.pi / 5) + np.sin(4 * np.pi / 5) + 1
     + np.sin(np.pi / 3) + np.sin(2 * np.pi / 3)
     + np.sin(2 * np.pi / 3 + 2 * np.pi / 5)) * 0.2,
])
A3 = np.array([
    -(0.5 + np.cos(np.pi / 3) + np.cos(np.pi / 6)
      + np.cos(4 * np.pi / 5 - np.pi / 3
               + np.cos(2 * np.pi / 5 - np.pi / 3))) * 0.2,
    A2[1] + np.sin(2 * np.pi / 3) * 0.2,
])
base_angles = np.array([0.0, 2 * np.pi / 3, -2 * np.pi / 3])
left_anchors = np.array([A1, A2, A3])
right_anchors = left_anchors + EA * np.column_stack((
    np.cos(base_angles), np.sin(base_angles)
))

def global_to_local(point_global, point_left, point_right):
    """Convert a global point to a 5-bar frame whose x-axis is left -> right."""
    direction = point_right - point_left
    phi = np.arctan2(direction[1], direction[0])
    rotation = np.array([
        [np.cos(phi), -np.sin(phi)],
        [np.sin(phi),  np.cos(phi)],
    ])
    return rotation.T @ (np.asarray(point_global) - point_left)

effectors_local = np.array([
    global_to_local(effectors_global[i], left_anchors[i], right_anchors[i])
    for i in range(3)
])

print("Coordonnées locales des effecteurs (m) :")
for i, point in enumerate(effectors_local, start=1):
    print(f"E{i}_local = ({point[0]:.6f}, {point[1]:.6f})")

Coordonnées locales des effecteurs (m) :
E1_local = (0.100000, 0.373205)
E2_local = (0.100000, 0.373205)
E3_local = (0.100000, 0.373205)


## 4 — Cinématique inverse géométrique d'une 5-barre

Pour un effecteur local connu `E`, le coude gauche `C` est à l'intersection des cercles `(A, AB)` et `(E, BC)`. Le coude droit `D` est à l'intersection de `(EA, DE)` et `(E, CD)`. On conserve toutes les combinaisons valides des deux intersections, au lieu de choisir une seule branche. Chaque chaîne est étiquetée **OUT** si son coude tourne dans le sens trigonométrique (produit vectoriel positif), **IN** si le sens est horaire. Cette convention est explicite et locale au repère de la jambe. On indique aussi le mode d'assemblage de la 5-barre (croisé ou non croisé), qui est distinct du mode de travail.

In [4]:
from itertools import product

GEOMETRY_TOL = 1e-10

def cross_2d(vector_0, vector_1):
    return vector_0[0] * vector_1[1] - vector_0[1] * vector_1[0]

def circle_intersections(center_0, radius_0, center_1, radius_1, label):
    """Return the two intersections of circles, or an empty list if unreachable."""
    center_0 = np.asarray(center_0, dtype=float)
    center_1 = np.asarray(center_1, dtype=float)
    delta = center_1 - center_0
    distance = np.linalg.norm(delta)

    if distance <= GEOMETRY_TOL:
        if np.isclose(radius_0, radius_1, atol=GEOMETRY_TOL, rtol=0):
            raise ValueError(f"{label}: cercles confondus; la solution est singulière.")
        return []
    if distance > radius_0 + radius_1 + GEOMETRY_TOL:
        return []
    if distance < abs(radius_0 - radius_1) - GEOMETRY_TOL:
        return []

    along = (radius_0**2 - radius_1**2 + distance**2) / (2 * distance)
    height_squared = radius_0**2 - along**2
    if height_squared <= GEOMETRY_TOL:
        raise ValueError(f"{label}: cercles tangents ou presque tangents; configuration singulière.")

    unit = delta / distance
    perpendicular = np.array([-unit[1], unit[0]])
    foot = center_0 + along * unit
    height = np.sqrt(height_squared)
    return [foot + height * perpendicular, foot - height * perpendicular]

def wrap_angle(angle):
    return (angle + np.pi) % (2 * np.pi) - np.pi

def solve_5bar_inverse_all(endpoint):
    """Return every non-singular 5-bar IK branch for one target endpoint."""
    endpoint = np.asarray(endpoint, dtype=float)
    left_elbows = circle_intersections(
        [0.0, 0.0], AB, endpoint, BC, "Coude gauche"
    )
    right_elbows = circle_intersections(
        [EA, 0.0], DE, endpoint, CD, "Coude droit"
    )
    if not left_elbows or not right_elbows:
        raise ValueError(
            f"Effecteur local {endpoint} inaccessible par une ou les deux chaînes de la 5-barre."
        )

    candidates = []
    for elbow_left in left_elbows:
        for elbow_right in right_elbows:
            side = cross_2d(elbow_right - elbow_left, endpoint - elbow_left)
            if abs(side) <= GEOMETRY_TOL:
                continue
            angles = np.array([
                np.arctan2(elbow_left[1], elbow_left[0]),
                np.arctan2(elbow_right[1], elbow_right[0] - EA),
            ])
            left_bend = cross_2d(elbow_left, endpoint - elbow_left)
            right_bend = cross_2d(elbow_right - np.array([EA, 0.0]),
                                  endpoint - elbow_right)
            candidates.append({
                "angles": angles,
                "elbow_left": elbow_left,
                "elbow_right": elbow_right,
                "left_mode": "OUT" if left_bend > 0 else "IN",
                "right_mode": "OUT" if right_bend > 0 else "IN",
            "assembly_mode": "+" if side > 0 else "-",
            })

    if not candidates:
        raise ValueError(f"Aucune configuration valide pour l'effecteur {endpoint}.")
    return candidates

## 5 — Résolution des trois jambes et vérification

Chaque 5-barre peut avoir jusqu'à quatre combinaisons de coudes (IN/OUT pour la chaîne gauche et droite). On forme ensuite toutes les combinaisons des trois jambes, jusqu'à `4^3 = 64` solutions de robot pour cette pose et ces orientations de connexion. Le tableau donne leurs modes et angles moteurs; les solutions sont ordonnées de la plus proche à la plus éloignée de `reference_angles`. Les collisions entre barres ne sont pas filtrées.

In [5]:
leg_candidates = [
    solve_5bar_inverse_all(effectors_local[i]) for i in range(3)
]

def local_to_global(point_local, point_left, angle):
    rotation = np.array([
        [np.cos(angle), -np.sin(angle)],
        [np.sin(angle),  np.cos(angle)],
    ])
    return point_left + rotation @ point_local

all_solutions = []
for leg_combination in product(*leg_candidates):
    angles = np.array([candidate["angles"] for candidate in leg_combination])
    error = np.sum(wrap_angle(angles - reference_angles) ** 2)
    all_solutions.append({
        "angles": angles,
        "branches": leg_combination,
        "reference_error": error,
    })
all_solutions.sort(key=lambda solution: solution["reference_error"])

if not all_solutions:
    raise ValueError("Aucune solution complète pour cette pose et ces angles de connexion.")

# Choisir le numéro à afficher dans la figure (indexation à partir de 1).
solution_to_plot = 1
if not 1 <= solution_to_plot <= len(all_solutions):
    raise ValueError(f"solution_to_plot doit être entre 1 et {len(all_solutions)}.")
selected_solution = all_solutions[solution_to_plot - 1]
joint_angles = selected_solution["angles"]
elbows_left_local = np.array([
    candidate["elbow_left"] for candidate in selected_solution["branches"]
])
elbows_right_local = np.array([
    candidate["elbow_right"] for candidate in selected_solution["branches"]
])
elbows_left_global = np.array([
    local_to_global(elbows_left_local[i], left_anchors[i], base_angles[i])
    for i in range(3)
])
elbows_right_global = np.array([
    local_to_global(elbows_right_local[i], left_anchors[i], base_angles[i])
    for i in range(3)
])

# Verify every listed branch against the link lengths and requested endpoint.
for solution in all_solutions:
    for i, candidate in enumerate(solution["branches"]):
        c = candidate["elbow_left"]
        d = candidate["elbow_right"]
        e = effectors_local[i]
        measured = np.array([
            np.linalg.norm(c), np.linalg.norm(e - c),
            np.linalg.norm(e - d), np.linalg.norm(d - [EA, 0.0]),
        ])
        if not np.allclose(measured, [AB, BC, CD, DE], atol=1e-9, rtol=0):
            raise AssertionError(f"Longueurs incorrectes, solution {solution}, jambe {i + 1}.")
        theta_left, theta_right = solution["angles"][i]
        c_from_angles = AB * np.array([np.cos(theta_left), np.sin(theta_left)])
        d_from_angles = np.array([EA, 0.0]) + DE * np.array([
            np.cos(theta_right), np.sin(theta_right)
        ])
        if not (np.allclose(c_from_angles, c, atol=1e-9, rtol=0)
                and np.allclose(d_from_angles, d, atol=1e-9, rtol=0)):
            raise AssertionError(f"Angles incorrects, solution {solution}, jambe {i + 1}.")

print(f"{len(all_solutions)} solutions complètes trouvées; "
      f"{[len(candidates) for candidates in leg_candidates]} branches par jambe.")
print("\nSolution | Modes de travail (gauche/droite) | Modes d'assemblage | Angles moteurs (degrés)")
for number, solution in enumerate(all_solutions, start=1):
    modes = " ; ".join(
        f"J{i + 1} {branch['left_mode']}/{branch['right_mode']}"
        for i, branch in enumerate(solution["branches"])
    )
    assembly = ", ".join(branch["assembly_mode"] for branch in solution["branches"])
    degrees = np.round(np.degrees(solution["angles"]).reshape(-1), 2)
    print(f"{number:02d} | {modes} | {assembly} | {degrees.tolist()}")

print(f"\nSolution affichée : {solution_to_plot} (la plus proche de la référence = 1).")
print("Tous les angles moteurs sont vérifiés par rapport aux longueurs de barres.")

27 solutions complètes trouvées; [3, 3, 3] branches par jambe.

Solution | Modes de travail (gauche/droite) | Modes d'assemblage | Angles moteurs (degrés)
01 | J1 IN/OUT ; J2 IN/OUT ; J3 IN/OUT | +, +, + | [90.0, 90.0, 90.0, 90.0, 90.0, 90.0]
02 | J1 OUT/OUT ; J2 IN/OUT ; J3 IN/OUT | +, +, + | [60.0, 90.0, 90.0, 90.0, 90.0, 90.0]
03 | J1 IN/IN ; J2 IN/OUT ; J3 IN/OUT | +, +, + | [90.0, 120.0, 90.0, 90.0, 90.0, 90.0]
04 | J1 IN/OUT ; J2 IN/OUT ; J3 IN/IN | +, +, + | [90.0, 90.0, 90.0, 90.0, 90.0, 120.0]
05 | J1 IN/OUT ; J2 IN/OUT ; J3 OUT/OUT | +, +, + | [90.0, 90.0, 90.0, 90.0, 60.0, 90.0]
06 | J1 IN/OUT ; J2 IN/IN ; J3 IN/OUT | +, +, + | [90.0, 90.0, 90.0, 120.0, 90.0, 90.0]
07 | J1 IN/OUT ; J2 OUT/OUT ; J3 IN/OUT | +, +, + | [90.0, 90.0, 60.0, 90.0, 90.0, 90.0]
08 | J1 OUT/OUT ; J2 IN/OUT ; J3 IN/IN | +, +, + | [60.0, 90.0, 90.0, 90.0, 90.0, 120.0]
09 | J1 OUT/OUT ; J2 IN/OUT ; J3 OUT/OUT | +, +, + | [60.0, 90.0, 90.0, 90.0, 60.0, 90.0]
10 | J1 IN/IN ; J2 IN/OUT ; J3 IN/IN | +, +, + 

## 6 — Affichage de la configuration inverse

Utiliser le curseur `solution_to_plot` pour parcourir les configurations du tableau; le dessin et les modes IN/OUT se mettent à jour automatiquement. Les modes d'assemblage `+` ou `-` sont les signes de `(D-C) × (E-C)`.

In [ ]:
import ipywidgets as widgets
from IPython.display import display

colors = ["tab:blue", "tab:orange", "tab:green"]

def plot_solution(solution_number):
    solution = all_solutions[solution_number - 1]
    fig, ax = plt.subplots(figsize=(8, 8))

    for i, color in enumerate(colors):
        left = left_anchors[i]
        right = right_anchors[i]
        vertex = platform_vertices[i]
        branch = solution["branches"][i]
        c = local_to_global(branch["elbow_left"], left, base_angles[i])
        d = local_to_global(branch["elbow_right"], left, base_angles[i])
        e = effectors_global[i]

        ax.plot([left[0], c[0], e[0], d[0], right[0]],
                [left[1], c[1], e[1], d[1], right[1]],
                "o-", color=color, linewidth=2, label=f"5-barre {i + 1}")
        ax.plot([e[0], vertex[0]], [e[1], vertex[1]], "--",
                color=color, linewidth=1.5)
        ax.scatter(*left, marker="s", color="black", zorder=4)
        ax.scatter(*right, marker="s", color="black", zorder=4)
        ax.text(e[0], e[1],
                f" E{i + 1} ({branch['left_mode']}/{branch['right_mode']})")

    closed_triangle = np.vstack((platform_vertices, platform_vertices[0]))
    ax.plot(closed_triangle[:, 0], closed_triangle[:, 1], "k-",
            linewidth=2, label="Triangle désiré")
    ax.scatter(platform_vertices[:, 0], platform_vertices[:, 1],
               marker="D", color="black", zorder=5)
    mode_summary = " | ".join(
        f"J{i + 1}:{branch['left_mode']}/{branch['right_mode']}"
        for i, branch in enumerate(solution["branches"])
    )
    ax.set_title(
        f"IK — solution {solution_number}/{len(all_solutions)} : {mode_summary}"
    )
    ax.set_xlabel("x (m)")
    ax.set_ylabel("y (m)")
    ax.axis("equal")
    ax.grid(True)
    ax.legend(loc="best")
    plt.show()
    plt.close(fig)

print("Solutions length: ", len(all_solutions))
solution_to_plot = widgets.IntSlider(
    value=1,
    min=1,
    max=len(all_solutions),
    step=1,
    description="Solution",
    continuous_update=False,
)
solution_output = widgets.interactive_output(
    plot_solution, {"solution_number": solution_to_plot}
)
display(widgets.VBox([solution_to_plot, solution_output]))

Solutions length:  27
